In [1]:
from pathlib import Path, PureWindowsPath
import gc
import gzip
import hashlib
import json
import os
import platform

import joblib
import lightgbm
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from joblib.externals import cloudpickle

ROOT = Path.cwd() / 'decision_plus_data' / 'outputs' / 'v11_prospective'
CACHE_2026 = os.environ.get('DECISION_V11_2026')
FREEZE_DIR = os.environ.get('DECISION_BOARD_FREEZE_DIR')
EXPORT_DIR = Path(os.environ.get('DECISION_PORTFOLIO_OUTPUT', str(ROOT / 'portfolio_2026'))).expanduser()

PRIMARY_OVERLAP = .10
MIN_BOARD_PITCHES, MIN_BOARD_GAMES = 300, 20
SEED, BOOT_DRAWS = 817, 1000
VERSIONS = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                sklearn=sklearn.__version__, lightgbm=lightgbm.__version__,
                scipy=scipy.__version__, joblib=joblib.__version__)

In [2]:
KEY = ['game_pk','at_bat_number','pitch_number']
RAW_COLS = ['game_pk','game_date','game_type','at_bat_number','pitch_number','batter','pitcher',
 'description','delta_run_exp','balls','strikes','outs_when_up','on_1b','on_2b','on_3b',
 'plate_x','plate_z','sz_top','sz_bot','pitch_type','release_speed','release_spin_rate',
 'release_extension','release_pos_x','release_pos_z','pfx_x','pfx_z','stand','p_throws',
 'inning','bat_score','fld_score','bat_score_diff','estimated_woba_using_speedangle','events']
OPTIONAL = {'inning','bat_score','fld_score','bat_score_diff','estimated_woba_using_speedangle','events','sz_top','sz_bot'}
REQUIRED = set(RAW_COLS) - OPTIONAL
SWING = {'swinging_strike','swinging_strike_blocked','foul','foul_tip','hit_into_play','hit_into_play_no_out','hit_into_play_score'}
TAKE = {'ball','blocked_ball','called_strike'}
EXTRA_FIELDS = ['inning_topbot','post_bat_score','home_score','away_score','post_home_score','post_away_score']
RAW_COLS += EXTRA_FIELDS
OPTIONAL.update(EXTRA_FIELDS)

In [3]:
def read_chunk(path,year):
    name=path.name.lower()
    if name.endswith('.pkl'): x=pd.read_pickle(path)
    elif name.endswith('.parquet'): x=pd.read_parquet(path)
    elif name.endswith('.csv.gz') or name.endswith('.csv'):x=pd.read_csv(path,usecols=lambda c:c in RAW_COLS,low_memory=False)
    else:raise ValueError('Unsupported local source file: '+str(path))
    if not isinstance(x,pd.DataFrame):raise ValueError('Not a DataFrame: '+str(path))
    missing=REQUIRED-set(x.columns)
    if missing:raise ValueError(f'{path.name}: missing required columns {sorted(missing)}')
    x=x.loc[:,[c for c in RAW_COLS if c in x]].copy()
    x['game_date']=pd.to_datetime(x.game_date,errors='coerce').dt.normalize()
    x=x.loc[x.game_date.dt.year.eq(year)&x.game_type.astype('string').eq('R')].copy()
    for c in RAW_COLS:
        if c not in x:x[c]=np.nan
    x['__source']=path.name
    return x

def read_set(paths,year):
    frames=[];log=[]
    for i,p in enumerate(paths,1):
        d=read_chunk(p,year)
        log.append({'year':year,'source':str(p),'rows_regular':len(d)})
        if len(d):frames.append(d)
        if i%15==0:print('Read',i,'/',len(paths),'source files')
    if not frames:raise ValueError(f'No {year} regular-season pitches in chosen source')
    raw=pd.concat(frames,ignore_index=True)
    del frames;gc.collect()
    return raw,pd.DataFrame(log)

def audit_unique(raw,year):
    for c in KEY:raw[c]=pd.to_numeric(raw[c],errors='coerce')
    if raw[KEY].isna().any().any():raise ValueError(f'{year} missing natural pitch IDs')
    raw=raw.sort_values('__source',kind='mergesort').reset_index(drop=True)
    repeated=raw.loc[raw.duplicated(KEY,keep=False)].copy()
    critical=['game_date','batter','pitcher','description','delta_run_exp','pitch_type',
              'balls','strikes','plate_x','plate_z','release_speed','pfx_x','pfx_z']
    if len(repeated):
        changed=pd.DataFrame({c:repeated.groupby(KEY,dropna=False,sort=False)[c].nunique(dropna=False).gt(1)
                              for c in critical})
        conflict_idx=changed.index[changed.any(axis=1)]
        conflict=pd.MultiIndex.from_frame(raw[KEY]).isin(conflict_idx)
        counts=changed.sum().rename('conflicting_keys').reset_index().rename(columns={'index':'field'})
    else:
        conflict=np.zeros(len(raw),bool);counts=pd.DataFrame(columns=['field','conflicting_keys'])
    cleaned=raw.loc[~conflict].drop_duplicates(KEY,keep='first').copy()
    dates=cleaned.game_date;games=cleaned.game_pk.nunique()
    report={'year':year,'input_rows':len(raw),'overlap_key_count':int(repeated[KEY].drop_duplicates().shape[0]),
            'conflicting_unique_keys':int(len(set(map(tuple,raw.loc[conflict,KEY].values)))),
            'cleaned_unique_pitches':len(cleaned),'games':int(games),
            'first_date':str(dates.min().date()),'last_date':str(dates.max().date()),
            'delta_run_exp_coverage':float(pd.to_numeric(cleaned.delta_run_exp,errors='coerce').notna().mean())}
    return cleaned,pd.DataFrame([report]),counts

In [4]:
def prepare(raw,year):
    d=raw.copy()
    before=len(d)
    d['description']=d.description.astype('string').str.strip().str.lower()
    d=d.loc[d.description.isin(SWING|TAKE)].copy()
    d['swung']=d.description.isin(SWING).astype('int8')
    numbers=['batter','pitcher','balls','strikes','outs_when_up','plate_x','plate_z','sz_top','sz_bot',
             'delta_run_exp','release_speed','release_spin_rate','release_extension','release_pos_x',
             'release_pos_z','pfx_x','pfx_z','inning','bat_score','fld_score',
             'bat_score_diff','estimated_woba_using_speedangle']
    for c in numbers:d[c]=pd.to_numeric(d[c],errors='coerce')
    d=d.dropna(subset=['batter','pitcher','balls','strikes','outs_when_up','plate_x','plate_z',
                       'delta_run_exp','game_date']).copy()
    d=d.loc[d.balls.between(0,3)&d.strikes.between(0,2)&d.outs_when_up.between(0,2)].copy()
    d['zone_valid']=(d.sz_top.notna()&d.sz_bot.notna()&(d.sz_top>d.sz_bot))
    d['inning']=d.inning.fillna(5)
    d['bat_score_diff']=d.bat_score_diff.fillna(d.bat_score-d.fld_score).fillna(0)
    d['base_state']=(d.on_1b.notna().astype('int8')+2*d.on_2b.notna().astype('int8')+
                     4*d.on_3b.notna().astype('int8'))
    d['two_strikes']=d.strikes.eq(2).astype('int8')
    for c in ['p_throws','stand','pitch_type']:d[c]=d[c].astype('string').fillna('Unknown')
    d['swing_outcome']=np.select([d.description.isin(['swinging_strike','swinging_strike_blocked']),
                        d.description.isin(['foul','foul_tip']),
                        d.description.isin(['hit_into_play','hit_into_play_no_out','hit_into_play_score'])],
                        ['whiff','foul','bip'],default='take')
    d['whiff']=d.swing_outcome.eq('whiff').astype('int8')
    d['batter']=d.batter.astype('int64');d['pitcher']=d.pitcher.astype('int64')
    d['estimated_woba_using_speedangle']=pd.to_numeric(d.estimated_woba_using_speedangle,errors='coerce')
    d['season']=year
    d['raw_x_band']=pd.cut(d.plate_x,[-np.inf,-1.5,-.83,0,.83,1.5,np.inf],
                           labels=['x<-1.5','-1.5 to -.83','-.83 to 0','0 to .83','.83 to 1.5','x>1.5']).astype(str)
    d['raw_z_band']=pd.cut(d.plate_z,[-np.inf,1.5,2.25,3.0,3.75,np.inf],
                           labels=['z<1.5','1.5 to 2.25','2.25 to 3.0','3.0 to 3.75','z>3.75']).astype(str)
    d['fixed_outer']=(d.plate_x.abs()>0.83)|(d.plate_z<1.5)|(d.plate_z>3.5)
    d=d.sort_values(['game_date']+KEY,kind='mergesort').reset_index(drop=True)
    return d,{'year':year,'source_unique':before,'clean_labeled':len(d),
              'zone_valid_n':int(d.zone_valid.sum()),'zone_invalid_kept':int((~d.zone_valid).sum()),
              'missing_sz_top_kept':int(d.sz_top.isna().sum()),
              'missing_sz_bot_kept':int(d.sz_bot.isna().sum()),
              'action_swing_rate':float(d.swung.mean())}

In [5]:
STATE = ['balls', 'strikes', 'outs_when_up', 'base_state']
HALF = ['game_pk', 'inning', 'half']
EVENT_MAP = {
    **{s: 'in_play' for s in SWING if s.startswith('hit_into_play')},
    'foul': 'foul', 'foul_tip': 'foul_tip',
    'swinging_strike': 'miss', 'swinging_strike_blocked': 'miss',
    'ball': 'ball', 'blocked_ball': 'ball', 'called_strike': 'called_strike',
}
SWING_EVENTS = ['miss', 'foul', 'foul_tip', 'in_play']
TAKE_EVENTS = ['ball', 'called_strike']

def transition_table(raw):
    x = raw.sort_values(KEY).copy()
    x['half'] = x.inning_topbot.astype('string').str.strip().str.lower().map(
        {'top': 0, 'bot': 1, 'bottom': 1}).astype('float64')
    x['base_state'] = sum(2**i * x[c].notna().astype(int) for i,c in enumerate(['on_1b','on_2b','on_3b']))
    for c in KEY[1:] + STATE + ['inning','bat_score','post_bat_score','home_score','away_score','post_home_score','post_away_score']:
        x[c] = pd.to_numeric(x[c], errors='coerce').astype('float64')
    top = x.half.eq(0)
    known_half = x.half.notna()
    pre_fallback = x.away_score.where(top, x.home_score).where(known_half)
    post_fallback = x.post_away_score.where(top, x.post_home_score).where(known_half)
    x['pre_score'] = x.bat_score.fillna(pre_fallback)
    x['post_score'] = x.post_bat_score.fillna(post_fallback)
    cols = HALF[1:] + ['at_bat_number','pitch_number','pre_score'] + STATE
    next_row = x.groupby('game_pk', sort=False)[cols].shift(-1)
    same_half = x[HALF[1:]].eq(next_row[HALF[1:]]).all(axis=1)
    same_pa = x.at_bat_number.eq(next_row.at_bat_number)
    next_half = (2 * next_row.inning + next_row.half).eq(2 * x.inning + x.half + 1)
    next_pa = (next_row.at_bat_number.eq(x.at_bat_number + 1)
               & next_row.pitch_number.eq(1) & next_row.balls.eq(0) & next_row.strikes.eq(0))
    x['half_end'] = ~same_half & next_half & next_pa & next_row.outs_when_up.eq(0)
    contiguous = (same_pa & next_row.pitch_number.eq(x.pitch_number + 1)) | (~same_pa & next_pa)
    score_ok = x.post_score.ge(x.pre_score) & x.post_score.notna() & x.pre_score.notna()
    state_ok = x.balls.between(0,3) & x.strikes.between(0,2) & x.outs_when_up.between(0,2)
    same_half_valid = same_half & contiguous & x.post_score.eq(next_row.pre_score)
    x['row_ok'] = score_ok & state_ok & (same_half_valid | x.half_end)
    g = x.groupby(HALF, sort=False, dropna=False)
    first_row = ~x.duplicated(HALF)
    valid_start = (x.pitch_number.eq(1) & x.balls.eq(0) & x.strikes.eq(0) & x.outs_when_up.eq(0))
    x['start_ok'] = ~first_row | valid_start
    x['usable_half'] = (g.row_ok.transform('all') & g.start_ok.transform('all')
                        & g.half_end.transform('any') & x.half.notna() & x.inning.notna())
    x['runs_pitch'] = x.post_score - x.pre_score
    x['future_runs'] = g.post_score.transform('last') - x.pre_score
    x['next_pa'] = ~same_pa & same_half
    for c in STATE:
        x['after_' + c] = next_row[c].where(same_half)
    return x[KEY + ['game_date','description','delta_run_exp','usable_half','half_end','next_pa',
                   'runs_pitch','future_runs'] + STATE + ['after_' + c for c in STATE]]

class Expectancy:
    def fit(self, history):
        d = history.loc[history.usable_half.fillna(False)]
        self.available = len(d) > 0
        if not self.available:
            return self
        state_mean = d.groupby(STATE).future_runs.agg(['sum','size'])
        base_mean = d.groupby(STATE[2:]).future_runs.mean()
        reference = base_mean.reindex(state_mean.index.droplevel([0,1])).to_numpy()
        self.table = (state_mean['sum'] + 20 * reference) / (state_mean['size'] + 20)
        self.base_mean = base_mean
        self.mean = float(d.future_runs.mean())
        return self

    def predict(self, state):
        idx = pd.MultiIndex.from_frame(state[STATE])
        a = self.table.reindex(idx).to_numpy()
        b = self.base_mean.reindex(idx.droplevel([0,1])).to_numpy()
        return np.where(np.isfinite(a), a, np.where(np.isfinite(b), b, self.mean))

    def reward(self, d):
        reward = np.full(len(d), np.nan)
        usable = d.usable_half.to_numpy(dtype=bool, na_value=False)
        if not self.available or not usable.any():
            return reward
        valid = d.loc[usable]
        half_end = valid.half_end.to_numpy(dtype=bool, na_value=False)
        after = valid[['after_' + c for c in STATE]].set_axis(STATE, axis=1)
        post = np.where(half_end, 0, self.predict(after))
        reward[usable] = valid.runs_pitch.to_numpy(dtype=float) + post - self.predict(valid)
        return reward

    def routine_value(self, d, event):
        after = d[STATE].astype('int64').copy()
        runs = np.zeros(len(d))
        if event == 'ball':
            walk = after.balls.eq(3)
            bases = after.base_state.to_numpy(dtype=int)
            forced = np.where((bases & 1) == 0, bases | 1,
                        np.where((bases & 2) == 0, bases | 3, 7))
            runs = (walk & (bases == 7)).astype(float).to_numpy()
            after.loc[walk, 'base_state'] = forced[walk]
            after['balls'] = (after.balls + 1).where(~walk, 0)
            after.loc[walk,'strikes'] = 0
        elif event == 'foul':
            after['strikes'] = (after.strikes + 1).clip(upper=2)
        else:
            strikeout = after.strikes.eq(2)
            after.loc[strikeout, 'outs_when_up'] += 1
            after['strikes'] = (after.strikes + 1).where(~strikeout, 0)
            after.loc[strikeout,'balls'] = 0
        post = np.where(after.outs_when_up.eq(3), 0, self.predict(after))
        return runs + post - self.predict(d)

In [6]:
def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(2**20),b''):
            h.update(block)
    return h.hexdigest()

CACHE_SUFFIXES = ('.csv.gz', '.csv', '.parquet', '.pkl')

def cache_files(path):
    if path.is_file() and path.name.lower().endswith(CACHE_SUFFIXES):
        return [path]
    if path.is_dir():
        return sorted(p for p in path.iterdir()
                      if p.is_file() and p.name.lower().startswith('statcast_')
                      and p.name.lower().endswith(CACHE_SUFFIXES))
    return []

def local_path(value):
    path = Path(value).expanduser()
    windows = PureWindowsPath(str(path))
    if os.name != 'nt' and windows.drive.endswith(':') and windows.is_absolute():
        mounted = Path('/mnt') / windows.drive[0].lower()
        if mounted.is_dir():
            return mounted.joinpath(*windows.parts[1:])
    return path

def previous_cache(year):
    report = ROOT.parent / 'v10_reward_sensitivity' / 'input_hashes.csv'
    if not report.is_file():
        return []
    hashes = pd.read_csv(report)
    if not {'year','file'}.issubset(hashes.columns):
        return []
    paths = [local_path(value) for value in hashes.loc[hashes.year.eq(year),'file'].dropna()]
    if paths and len(set(paths)) == len(paths) and all(path.is_file() for path in paths):
        return sorted(paths)
    return []

def cache_candidates(year):
    home, cwd = Path.home(), Path.cwd()
    roots = [cwd, *cwd.parents, home, home/'Documents', home/'Downloads',
             home/'Desktop', home/'OneDrive'/'Documents']
    roots.extend(Path(os.environ[key])/'Documents' for key in ('OneDrive','OneDriveConsumer','OneDriveCommercial')
                 if os.environ.get(key))
    repo = 'mlb-pitcher-dashboard-flask'
    relative = Path('data')/'cache'/'statcast'/str(year)
    for root in roots:
        if root == root.parent:
            continue
        yield root/relative
        yield root/repo/relative
        yield root/'GitHub'/repo/relative
        yield root/'statcast'/str(year)
        yield root/'statcast_cache'/str(year)

def resolve_cache_files(cache, year):
    if cache:
        configured = local_path(cache)
        if files := cache_files(configured):
            return files
        raise FileNotFoundError(
            f'CACHE_{year} points to {configured}, where no supported Statcast files were found. '
            f'Set it to the cache folder or exact file, or set CACHE_{year} = None to search automatically. '
            f'Notebook working directory: {Path.cwd()}')
    if files := previous_cache(year):
        return files
    matches = {}
    for folder in cache_candidates(year):
        if files := cache_files(folder):
            matches[folder.resolve()] = files
    if len(matches) == 1:
        return next(iter(matches.values()))
    if matches:
        options = '\n'.join(f'  {path} ({len(files)} files)' for path, files in matches.items())
        raise FileNotFoundError(f'Multiple {year} caches found. Set CACHE_{year} to the one used in V10:\n{options}')
    raise FileNotFoundError(
        f'No {year} Statcast cache found from the saved V10 inputs or nearby project folders. '
        f'Copy the cache path from V10 or File Explorer and set CACHE_{year} = Path(r"your path") '
        f'in the first code cell. Notebook working directory: {Path.cwd()}')

def load_season(cache, year):
    files = resolve_cache_files(cache, year)
    print(f'{year} cache: {files[0].parent} ({len(files)} files)')
    raw, _ = read_set(files,year)
    clean, audit, conflicts = audit_unique(raw,year)
    del raw
    gc.collect()
    return clean, audit, conflicts, [dict(file=str(p),sha256=digest(p)) for p in files]

def with_rewards(raw, cohort, expectancy):
    t = transition_table(raw)
    t['reconstructed_rv'] = expectancy.reward(t)
    d = cohort.merge(t[KEY+['reconstructed_rv']],on=KEY,how='left',validate='one_to_one')
    d['event_v9'] = d.description.map(EVENT_MAP)
    return d

In [7]:
from IPython.display import HTML

VERIFIED_NAMES = {543807: 'George Springer',
 621493: 'Taylor Ward',
 622268: 'Donovan Walton',
 641343: 'Jake Bauers',
 650402: 'Gleyber Torres',
 657041: 'Lane Thomas',
 660844: 'Leo Rivas',
 663656: 'Kyle Tucker',
 665742: 'Juan Soto',
 666397: 'Edouard Julien',
 668709: 'JJ Bleday',
 668804: 'Bryan Reynolds',
 668885: 'Austin Martin',
 669257: 'Will Smith',
 669364: 'Xavier Edwards',
 676914: 'Davis Schneider',
 678011: 'Anthony Seigler',
 678246: 'Miguel Vargas',
 681393: 'Connor Norby',
 681715: 'Heriberto Hernández',
 694497: 'Evan Carter',
 695670: 'Harry Ford',
 699013: 'Esmerlyn Valdez',
 805300: 'Jakob Marsee',
 805808: 'Kevin McGonigle'}

from html import escape
from urllib.error import URLError
from urllib.request import urlopen

def resolve_player_names(batter_ids):
    names = VERIFIED_NAMES.copy()
    missing = sorted(set(map(int, batter_ids)) - names.keys())
    for start in range(0, len(missing), 100):
        batch = missing[start:start + 100]
        url = 'https://statsapi.mlb.com/api/v1/people?personIds=' + ','.join(map(str, batch))
        try:
            with urlopen(url, timeout=10) as response:
                people = json.load(response)['people']
        except (URLError, TimeoutError, ValueError, KeyError) as exc:
            print(f'MLB name lookup unavailable. Unresolved hitters will display by ID ({exc}).')
            break
        names.update({player['id']: player['fullName'] for player in people
                      if player.get('id') in batch and player.get('fullName')})
    return names


def leaderboard_html(board, supported, eligible):
    leaders = board.loc[board.qualified].head(25)
    rows = []
    for row in leaders.itertuples(index=False):
        player = escape(row.player_name)
        cells = [
            str(int(row.rank)),
            f'<a href="https://www.mlb.com/player/{int(row.batter)}"><strong>{player}</strong></a>',
            f'<strong>{row.decision_runs_per_100:+.3f}</strong>',
            f'{int(row.supported_pitches):,}',
            f'{row.supported_pct:.1f}%',
            f'{row.low_95:+.3f} to {row.high_95:+.3f}',
        ]
        rows.append('<tr>' + ''.join(
            f'<td align="{("left" if index == 1 else "right")}">{value}</td>'
            for index, value in enumerate(cells)
        ) + '</tr>')
    headers = ['Rank', 'Hitter', 'Decision Runs / 100', 'Supported Pitches', 'Coverage', '95% Interval']
    return (
        '<h2>Decision+ | 2026 MLB Leaderboard</h2>'
        '<p><strong>Regular season · March 25–September 27, 2026</strong></p>'
        f'<p>{supported:,} supported pitches of {eligible:,} eligible '
        f'({100 * supported / eligible:.1f}%) · Minimum 300 supported pitches and 20 games</p>'
        '<table><thead><tr>' + ''.join(f'<th>{label}</th>' for label in headers)
        + '</tr></thead><tbody>' + ''.join(rows) + '</tbody></table>'
        '<p><small>Decision Runs / 100 estimates the value of a hitter’s swing and take choices '
        'relative to league swing behavior. Coverage is the share of eligible pitches used for each hitter. '
        'Intervals resample games with the model held fixed. This retrospective board includes the April–May '
        'training window and uses final pitch location.</small></p>'
    )

In [8]:
def resolve_freeze_dir():
    if FREEZE_DIR:
        candidates = [local_path(FREEZE_DIR)]
    else:
        working = Path.cwd()
        roots = [working, *working.parents, Path.home() / 'Downloads', Path.home() / 'Documents']
        candidates = [root / 'decision_plus_data' / 'outputs' / 'v11_prospective' / 'frozen_2026'
                      for root in roots if root != root.parent]
    for folder in candidates:
        if (folder / 'protocol.json').is_file() and (folder / 'models.pkl.gz').is_file():
            return folder
    locations = '\n'.join(f'  {folder}' for folder in dict.fromkeys(candidates))
    raise FileNotFoundError(
        'Set FREEZE_DIR in the first code cell to the V11 folder containing '
        f'protocol.json and models.pkl.gz. Searched:\n{locations}')


def load_models():
    folder = resolve_freeze_dir()
    protocol_path = folder / 'protocol.json'
    model_path = folder / 'models.pkl.gz'
    protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
    if protocol['synthetic'] or protocol['rules']['primary_overlap'] != PRIMARY_OVERLAP:
        raise ValueError('Use the production V11 freeze with the 0.10 overlap cutoff.')
    if digest(model_path) != protocol['bundle_sha256']:
        raise ValueError('The saved model checksum differs from the V11 protocol.')
    versions = {name: (value, VERSIONS.get(name)) for name, value in protocol['versions'].items()
                if VERSIONS.get(name) != value}
    if versions:
        raise RuntimeError(f'Software versions differ from the freeze (frozen, current): {versions}')
    with gzip.open(model_path, 'rb') as file:
        return protocol, cloudpickle.load(file)


def score_season(protocol, bundle):
    if CACHE_2026:
        raw, audit, _, sources = load_season(CACHE_2026, 2026)
    else:
        recorded_paths = [local_path(item['file']) for item in protocol['source_2026']]
        if recorded_paths and all(path.is_file() for path in recorded_paths):
            raw, _ = read_set(recorded_paths, 2026)
            raw, audit, _ = audit_unique(raw, 2026)
            sources = [dict(file=str(path), sha256=digest(path)) for path in recorded_paths]
        else:
            raw, audit, _, sources = load_season(None, 2026)
    def signature(files):
        return sorted((PureWindowsPath(item['file']).name, item['sha256']) for item in files)
    if signature(sources) != signature(protocol['source_2026']):
        raise ValueError('The cache names or file checksums differ from the V11 freeze.')
    eligible, _ = prepare(raw, 2026)
    pitches = with_rewards(raw, eligible, bundle['expectancy'])
    features = bundle['ref_features'].transform(pitches)
    reference = bundle['reference'].predict_proba(features)[:, 1]
    pitches['league_swing_probability'] = np.clip(reference, .02, .98)
    pitches['swing_propensity'] = bundle['behavior'].predict(pitches)
    pitches['swing_value'], pitches['take_value'] = bundle['candidates']['direct_savant'].predict(pitches)
    pitches['swing_take_gap'] = pitches.swing_value - pitches.take_value
    pitches['decision_runs'] = (pitches.swung - pitches.league_swing_probability) * pitches.swing_take_gap
    pitches['within_overlap'] = pitches.swing_propensity.between(PRIMARY_OVERLAP, 1 - PRIMARY_OVERLAP)
    pitches['has_reconstructed_label'] = pitches.reconstructed_rv.notna()
    pitches['supported'] = pitches.within_overlap & pitches.has_reconstructed_label
    pitches['april_may_fit_window'] = pitches.game_date.dt.month.isin([4, 5])
    if pitches.duplicated(KEY).any():
        raise ValueError('Expected unique game, plate appearance, and pitch keys.')
    return pitches, audit


def coverage_counts(pitches):
    overlap, label = pitches.within_overlap, pitches.has_reconstructed_label
    return dict(eligible=len(pitches), supported=int((overlap & label).sum()),
                overlap_only=int(overlap.sum()),
                excluded_overlap_only=int((~overlap & label).sum()),
                excluded_label_only=int((overlap & ~label).sum()),
                excluded_both=int((~overlap & ~label).sum()),
                supported_pct=100 * (overlap & label).mean())


def make_leaderboard(pitches, support_column):
    eligible = pitches.groupby('batter').size().rename('eligible_pitches')
    supported = pitches.loc[pitches[support_column]]
    game = supported.groupby(['batter','game_pk']).agg(
        pitches=('decision_runs','size'), runs=('decision_runs','sum'),
        fit_window=('april_may_fit_window','sum')).reset_index()
    board = game.groupby('batter').agg(
        supported_pitches=('pitches','sum'), supported_games=('game_pk','size'),
        decision_runs=('runs','sum'), april_may_supported_pitches=('fit_window','sum'))
    board = eligible.to_frame().join(board)
    counts = ['supported_pitches','supported_games','april_may_supported_pitches']
    board[counts] = board[counts].fillna(0).astype(int)
    board['supported_pct'] = 100*board.supported_pitches/board.eligible_pitches
    board['decision_runs_per_100'] = 100*board.decision_runs/board.supported_pitches.replace(0,np.nan)
    board['qualified'] = (board.supported_pitches>=MIN_BOARD_PITCHES) & (board.supported_games>=MIN_BOARD_GAMES)
    board['rank'] = board.decision_runs_per_100.where(board.qualified).rank(ascending=False,method='min').astype('Int64')
    rng = np.random.default_rng(SEED)
    board['low_95'], board['high_95'] = np.nan, np.nan
    for batter,g in game.groupby('batter',sort=True):
        if not board.at[batter,'qualified']:
            continue
        draws = rng.integers(0,len(g),size=(BOOT_DRAWS,len(g)))
        scores = 100*g.runs.to_numpy()[draws].sum(axis=1)/g.pitches.to_numpy()[draws].sum(axis=1)
        board.loc[batter,['low_95','high_95']] = np.quantile(scores,[.025,.975])
    return board.reset_index().sort_values(['rank','batter'],na_position='last')

protocol, bundle = load_models()
pitches, source_audit = score_season(protocol, bundle)
coverage = pd.DataFrame([
    {'period': 'season', **coverage_counts(pitches)},
    *({'period': str(month), **coverage_counts(group)}
      for month, group in pitches.groupby(pitches.game_date.dt.to_period('M'))),
])
season = coverage.iloc[0]
september = coverage.loc[coverage.period.eq('2026-09'), 'supported'].iloc[0]
if (season.eligible, season.supported, september) != (699092, 463184, 59573):
    raise RuntimeError('The run did not reproduce the published 2026 coverage counts.')

leaderboard = make_leaderboard(pitches, 'supported')
overlap_board = make_leaderboard(pitches, 'within_overlap')
qualified = leaderboard.loc[leaderboard.qualified].merge(
    overlap_board.loc[overlap_board.qualified], on='batter', suffixes=('_published', '_overlap'))
score_change = qualified.decision_runs_per_100_overlap - qualified.decision_runs_per_100_published
rank_change = qualified.rank_overlap - qualified.rank_published
def top25(board):
    return set(board.loc[board["rank"].le(25), "batter"])

sensitivity = pd.DataFrame([{
    'shared_qualified_hitters': len(qualified),
    'spearman': qualified.decision_runs_per_100_published.corr(
        qualified.decision_runs_per_100_overlap, method='spearman'),
    'median_abs_score_change': score_change.abs().median(),
    'median_abs_rank_change': rank_change.abs().median(),
    'top25_in_common': len(top25(leaderboard) & top25(overlap_board)),
}])

player_names = resolve_player_names(leaderboard.batter)
for table in (leaderboard, overlap_board):
    names = table.batter.map(player_names)
    table.insert(1, 'player_name', names.fillna('MLB ID ' + table.batter.astype(str)))

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for name, table in {
    'leaderboard_2026': leaderboard,
    'leaderboard_2026_overlap_only': overlap_board,
    'coverage_audit_2026': coverage,
    'support_sensitivity_2026': sensitivity,
    'cache_audit_2026': source_audit,
}.items():
    table.to_csv(EXPORT_DIR / f'{name}.csv', index=False)

pitch_columns = KEY + [
    'game_date', 'batter', 'pitcher', 'stand', 'p_throws', 'balls', 'strikes', 'pitch_type',
    'plate_x', 'plate_z', 'swung', 'swing_value', 'take_value', 'swing_take_gap',
    'league_swing_probability', 'swing_propensity', 'decision_runs', 'supported',
    'within_overlap', 'has_reconstructed_label', 'april_may_fit_window',
]
pitches[pitch_columns].to_csv(EXPORT_DIR / 'pitch_review_2026.csv.gz', index=False)
metadata = dict(season=2026, bundle_sha256=protocol['bundle_sha256'],
                overlap=[PRIMARY_OVERLAP, 1 - PRIMARY_OVERLAP],
                requires_reconstructed_label=True,
                minimum_supported_pitches=MIN_BOARD_PITCHES, minimum_supported_games=MIN_BOARD_GAMES,
                scope='retrospective, includes the April–May model fit window')
(EXPORT_DIR / 'export_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

display(HTML(leaderboard_html(leaderboard, int(season.supported), int(season.eligible))))

Read 15 / 41 source files
Read 30 / 41 source files


Rank,Hitter,Decision Runs / 100,Supported Pitches,Coverage,95% Interval
1,Taylor Ward,+0.993,"1,678",67.1%,+0.782 to +1.232
2,Gleyber Torres,+0.934,"1,170",67.2%,+0.626 to +1.249
3,Austin Martin,+0.920,893,65.8%,+0.570 to +1.278
4,JJ Bleday,+0.855,"1,341",64.0%,+0.589 to +1.096
5,Kevin McGonigle,+0.828,"1,835",66.3%,+0.582 to +1.095
6,Donovan Walton,+0.807,671,68.5%,+0.398 to +1.227
7,Edouard Julien,+0.805,698,61.5%,+0.368 to +1.228
8,Heriberto Hernández,+0.761,"1,253",64.3%,+0.513 to +1.022
9,Davis Schneider,+0.742,458,64.9%,+0.203 to +1.210
10,George Springer,+0.731,"1,336",67.6%,+0.479 to +0.989
